# 🐼 Pandas Mastery: The Data Wrangling Machine

## *"From Messy CSVs to Model-Ready Matrices"*

This notebook covers **90% of the Pandas workflows** needed for Real-World Machine Learning projects.

---

### 📚 Course Structure

| Level | Title | Key Concepts |
|-------|-------|-------------|
| 🟢 **1** | **The Explorer** | `read_csv`, `index_col`, `info`, `describe`, `loc` vs `iloc` |
| 🟡 **2** | **The Cleaner** | Missing Values (`fillna` strategies), Filtering, Types, Duplicates |
| 🔴 **3** | **The Engineer** | `apply` (no loops), `map`, `get_dummies`, `groupby` |
| 🟣 **4** | **The Wrangler** | Merging (`join`), Binning (`cut`), Time Series (`dt` accessors) |

---


In [ ]:
import pandas as pd
import numpy as np
print(f"✅ Pandas Version: {pd.__version__}")

# 🟢 LEVEL 1: The Explorer

**Goal:** Load data instantly and understand its structure without opening Excel.

## 1.1 Loading Data (The Right Way)

- `read_csv` is the workhorse.
- `index_col`: Use a column as the index (row labels) immediately.

In [ ]:
# Simulate creating a CSV file
data = """
ID,Name,Age,Salary,Dept,JoinDate
101,Alice,25,50000,IT,2023-01-15
102,Bob,30,60000,HR,2022-05-20
103,Charlie,35,70000,IT,2021-08-10
"""
with open('employee.csv', 'w') as f:
    f.write(data.strip())

# Load it!
df = pd.read_csv('employee.csv', index_col='ID')
print("DataFrame Loading:")
print(df.head())

## 1.2 Inspection Toolkit

Before you touch data, look at it.

1. `df.shape`: (Rows, Columns)
2. `df.info()`: Data types & Missing values
3. `df.describe()`: Statistical summary

In [ ]:
print(f"Shape: {df.shape}")
print("-"*20)
df.info()
print("-"*20)
print(df.describe())

## 1.3 Selecting Data: `loc` vs `iloc`

**The Golden Rule:**
- **`loc`** = **L**abel based (Rows by Index Name, Cols by Name)
- **`iloc`** = **I**nteger Position based (Row #, Col #)

In [ ]:
# loc (Label)
print(f"Alice's Salary: {df.loc[101, 'Salary']}")

# iloc (Position)
print(f"Row 0, Col 2: {df.iloc[0, 2]}") # Alice's Salary (Col 2 is Salary)

# Slicing
print("\nIT Department subset:")
print(df.loc[:, ['Name', 'Dept']])

# 🟡 LEVEL 2: The Cleaner

**Goal:** Fix the mess. Real data is full of holes, duplicates, and wrong types.

## 2.1 Handling Missing Data (`NaN`)

Models crashe on NaNs. You must **Drop** or **Fill**.

In [ ]:
messy = pd.DataFrame({
    'A': [1, 2, np.nan],
    'B': [5, np.nan, np.nan],
    'C': [1, 2, 3]
})

print(f"Missing count:\n{messy.isnull().sum()}")

# Strategy 1: Drop rows with ANY NaNs
clean_drop = messy.dropna()

# Strategy 2: Fill (Imputation)
messy['A'] = messy['A'].fillna(messy['A'].mean())  # Fill with Mean
messy['B'] = messy['B'].fillna(0)                  # Fill with Constant

print("\nFilled DataFrame:")
print(messy)

## 2.2 Boolean Filtering

Select rows based on conditions. **No loops needed.**

In [ ]:
# Get High Earners in IT
high_earners = df[(df['Salary'] > 55000) & (df['Dept'] == 'IT')]
print("High Earners in IT:")
print(high_earners)

# 🔴 LEVEL 3: The Engineer

**Goal:** Create features. Models need numbers, not strings.

## 3.1 Encoding Categorical Data

1. **`map`**: For Order/Binary (Male=0, Female=1)
2. **`get_dummies`**: For Categories (Red, Green, Blue)

In [ ]:
# Mapping (Binary/Ordinal)
df['Is_IT'] = df['Dept'].map({'IT': 1, 'HR': 0})

# One-Hot Encoding (Nominal)
dummies = pd.get_dummies(df['Dept'], prefix='Dept')
df = pd.concat([df, dummies], axis=1)

print(df.head())

## 3.2 The `apply` Function

Run ANY Python function on every row/column. Slower than vectorization, but super flexible.

In [ ]:
def tax_bracket(salary):
    if salary > 65000:
        return 'High'
    return 'Low'

df['Tax_Bracket'] = df['Salary'].apply(tax_bracket)
print(df[['Salary', 'Tax_Bracket']])

## 3.3 GroupBy (Pivot Tables in Code)

Aggregate data to find patterns.

In [ ]:
print("Average Salary by Dept:")
print(df.groupby('Dept')['Salary'].mean())

# 🟣 LEVEL 4: The Wrangler (Advanced)

**Goal:** Complex data manipulation needed for real-world pipelines.

## 4.1 Merging (Joins)

Combining multiple tables (SQL styles: Left, Inner, Outer).

In [ ]:
users = pd.DataFrame({'id': [1, 2], 'name': ['Alice', 'Bob']})
emails = pd.DataFrame({'id': [1, 2], 'email': ['a@x.com', 'b@x.com']})

# Merge on ID
merged = pd.merge(users, emails, on='id', how='inner')
print(merged)

## 4.2 Binning (`cut` / `qcut`)

Turning continuous variables (Age, Price) into categories (Young/Old, Low/High).

In [ ]:
ages = np.random.randint(0, 100, 10)
age_df = pd.DataFrame({'Age': ages})

# Cut into 3 bins (Equla Width)
age_df['Group'] = pd.cut(age_df['Age'], bins=3, labels=['Low', 'Med', 'High'])
print(age_df.head())

## 4.3 Time Series Essentials

Parsing dates and extracting features (Month, Day, Hour).

In [ ]:
df['JoinDate'] = pd.to_datetime(df['JoinDate'])

df['Year'] = df['JoinDate'].dt.year
df['Month'] = df['JoinDate'].dt.month_name()

print(df[['JoinDate', 'Year', 'Month']])

---
### 🏆 Final Challenge: Full Pipeline Function

A clean function taking a raw CSV path and returning `X_train`, `y_train`.

In [ ]:
def process_data(file_path):
    # 1. Load
    df = pd.read_csv(file_path, index_col='ID')
    
    # 2. Clean (Fill NaNs)
    df['Salary'] = df['Salary'].fillna(df['Salary'].mean())
    
    # 3. Engineer (Encode)
    df['Is_IT'] = (df['Dept'] == 'IT').astype(int)
    
    # 4. Features & Target
    features = ['Age', 'Salary', 'Is_IT']
    X = df[features].values
    return X

X_final = process_data('employee.csv')
print(f"Final Matrix Shape: {X_final.shape}")

In [ ]:
# Cleanup
import os
if os.path.exists('employee.csv'):
    os.remove('employee.csv')